# Extensión de la Ontología de Pizzas y razonamiento semántico en pizzerías

El objetivo de esta práctica es extender una Ontología estándar de pizzas en OWL para dar soporte al modelo de negocio de una pizzería.

El fichero con la Ontología, llamado `pizza.owl`, lo puedes encontrar en PoliformaT.

## Descripción del problema

La Ontología base `pizza.owl` únicamente describe el dominio alimentario (tipos de masa, toppings y categorías de pizza).

Para adaptar este modelo a la gestión real de un establecimiento, es necesario incorporar el modelo transaccional y de clientes.

El objetivo principal es modelar reglas de negocio explícitas e implícitas mediante Lógica de Descripción (DL) en Owlready2, de manera que el motor de razonamiento semántico (Pellet) pueda clasificar automáticamente a los clientes dentro de programas de fidelización (como *VIP* o *Preferente*) e inferir categorías de pedidos sin necesidad de programar condicionales manuales en Python.

## Estructura del trabajo:
* Parte 1: Extensión del modelo (TBox) con Owlready2.
* Parte 2: Poblado de datos (ABox) con Owlready2.
* Parte 3: Razonamiento semántico, incosistencia y consultas en Owlready2.


## Instalación e importación de Owlready2

In [ ]:
%pip install owlready2

In [1]:
from owlready2 import *
import subprocess
subprocess.run(['java', '-version'])

CompletedProcess(args=['java', '-version'], returncode=0)

## Visualización gráfica de Ontología

Con el objetivo de facilitaros la inspección del esquema conceptual (TBox), os proporcionamos un fragmento de código que analiza la Ontología y genera un diagrama visual en formato `.png`.

**Antes de ejecutarlo**: Aseguraos de haber cargado previamente la Ontología en memoria (objeto `onto`).

Al ejecutar el bloque de código, la imagen se guardará automáticamente en vuestro directorio de trabajo.


In [4]:
# Requiere: %pip install owlready2 graphviz
import graphviz
from owlready2 import *

########   1. CARGAR la ontología   ########
onto = get_ontology("pizza.owl")
############################################

# 2. Configurar el grafo con orientacion UML (de abajo a arriba)
dot = graphviz.Digraph('PizzaOntology', comment='Diagrama de Clases Pizza', format='png')
dot.attr(rankdir='BT', size='50,50')

# 3. Dibujar nodos y jerarquías
for cls in onto.classes():
    # Estilo especial según la categoría principal
    if cls in [onto.Pizza, onto.PizzaBase, onto.PizzaTopping, onto.Country]:
        dot.node(cls.name, shape='rectangle', style='filled', fillcolor='#ffcccb')
    else:
        dot.node(cls.name, shape='rectangle', style='filled', fillcolor='#e1f5fe')

    # Subclases (subClassOf)
    for parent in cls.is_a:
        if isinstance(parent, ThingClass):
            dot.edge(cls.name, parent.name, label='subClassOf')

# 4. Exportar y renderizar
output_path = dot.render('diagrama_pizza', view=False)
print(f'Imagen generada con éxito en: {output_path}')

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

## **Parte 1**: Extensión del Modelo (TBox) con Owlready2

En esta primera parte se extenderá la estructura conceptual de la Ontología definiendo las nuevas entidades, sus relaciones y restricciones lógicas, para posteriormente crear las instancias de prueba utilizando las clases y sintaxis nativas de Owlready2. Se pide:

1. Cargar el fichero `pizza.owl`.
2. Crear dentro del contexto de la Ontología las clases base Persona, Cliente (subclase de Persona) y Pedido.
3. Definir las siguientes propiedades de objeto estableciendo explícitamente sus dominios, rangos y propiedades inversas:
    * `realizaPedido`: que implica que un cliente realiza un pedido.
    * `esRealizadoPor`: inversa de `realizaPedido`.
    * `contienePizza`: que indica que un pedido contiene una pizza.
    * `esPizzaDe`: inversa de `contienePizza`.
4. Definir las siguientes clases equivalentes:
    * `PedidoGrande`: todo `Pedido`que contiene al menos 3 pizzas.
    * `ClienteVIP`: todo `Cliente` que realiza al menos un `PedidoGrande`.
    * `ClientePreferente`: todo `Cliente` que realiza al menos 2 pedidos.

5. Declarar la disyunción explícita entre clases raíz:  `Persona`, `Pedido`, `Pizza`, `PizzaBase` y `PizzaTopping`.

Al finalizar esta parte puedes visualizar la Ontología para comprobar que se han incluido las clases base.

## Nota: sintaxis de restricciones sobre propiedades en Owlready2

**1. Restricciones de cuantificación**:
  * **`propiedad.some(Clase)` (existencial, $\exists$)**: exige que exista al menos una relación con un elemento de esa clase.
  * **`propiedad.only(Clase)` (universal, $\forall$)**: obliga a que todas las relaciones existentes sean exclusivamente de esa clase.

**2. Restricciones de cardinalidad**:
  * **`propiedad.min(N, Clase)` ($\ge N$)**: exige relacionarse con al menos $N$ instancias.
  * **`propiedad.max(N, Clase)` ($\le N$)**: limita la relación a como máximo $N$ instancias.
  * **`propiedad.exactly(N, Clase)` ($= N$)**: obliga a relacionarse con exactamente $N$ instancias.

## **Parte 2**: Poblado de instancias (ABox) con Owlready2

En esta segunda parte se definirán nuevas instancias específicas del dominio que se está modelando. En concreto, se pide crear las siguientes instancias sin asignar explícitamente las clases `ClienteVIP` o `ClientePreferente`, ya que deberán ser inferidas posteriormente por el razonador semántico:

1. Pizzas personalizadas:
    * `pizza_custom_1`: tiene una base `ThinAndCrispyBase` y topping `MozzarellaTopping`.
    * `pizza_custom_2`: tiene una base `ThinAndCrispyBase` y topping `OliveTopping`.
    * `pizza_custom_3`: tiene una base `ThinAndCrispyBase` y topping `ParmesanTopping`.

2. Cliente 1 (`juan_cano`):
    * Realiza un único pedido (`pedido_101`).
    * `pedido_101` contiene 3 pizzas: una `MargheritaPizza`, una `AmericanPizza` y la pizza personalizada (`pizza_custom_1`).

3. Cliente 2 (`ana_uceda`):
    * Realiza dos pedidos independientes (`pedido_102` y `pedido_103`).
    * `pedido_102` contiene 1 pizza `SohoPizza`.
    * `pedido_103` contiene 1 pizza `MargheritaPizza`.

2. Cliente 3 (`carlos_puente`):
    * Realiza un único pedido (`pedido_104`).
    * `pedido_104` contiene 3 pizzas: una `SohoPizza`, una `AmericanHotPizza` y la pizza personalizada (`pizza_custom_2`).

3. Cliente 4 (`maria_perez`):
    * Realiza dos pedidos independientes (`pedido_105` y `pedido_106`).
    * `pedido_105` contiene 1 pizza `MargheritaPizza`.
    * `pedido_106` contiene la pizza personalizada `pizza_custom_3`.



## **Parte 3**: Razonamiento semántico, inconsistencia y consultas en Owlready2

En esta tercera parte se invocará el razonador desde Owlready2 para comprobar el proceso de inferencia semántica, se realizarán consultas sobre los datos inferidos y se validará el control de inconsistencias lógicas capturando excepciones propias de Owlready2. Se piden lon siguientes 3 apartados:

**1. Invocación del razonador e inferencia**
  * Ejecutar el razonador semántico (`sync_reasoner_pellet()`) de Owlready2.
  * Imprimir por pantalla la clase inferida para `pedido_101`.
  * Imprimir por pantalla las clases inferidas para `juan_cano`.
  * Imprimir por pantalla las clases inferidas para `ana_uceda`.
  * Verificar si `pizza_custom_1` fue clasificada automáticamente como `CheesyPizza`.

Ejecuta el razonador dentro de un bloque `try-except`. Capturar específicamente la excepción `OwlReadyInconsistentOntologyError` que lanza `Owlready2` cuando detecta un modelo incoherente, mostrando un mensaje explicativo por pantalla.


**2. Consultas**
  * Escribir y ejecutar una consulta sobre la Ontología razonada que devuelva el nombre de todos los clientes inferidos como `ClienteVIP` junto con los identificadores de sus pedidos.
  * Escribir y ejecutar una consulta sobre la Ontología razonada que devuelva el nombre de todos los clientes inferidos como `ClientePreferente` junto con los identificadores de sus pedidos.
  * Escribir y ejecutar una consulta sobre la Ontología razonada que devuelva las pizzas que han sido clasificacdas como `CheesyPizza` junto con los clientes y los identificadores de pedidos en los que se pidieron.
  * Escribir y ejecutar una consulta sobre la Ontología razondad que devuelvan los identificadores de pedidos con pizzas preparadas con una masa `ThinAndCrispyBase`.

**Nota: método `onto.search()` en Olwready2 para realizar consultas**:

Para realizar consultas a la ontología desde Python, Owlready2 ofrece el método `onto.search()`. Este método explora la base de conocimiento y devuelve una lista de Python con los objetos que cumplen los criterios especificados.

Parámetros habituales de búsqueda:

* `type`: filtra entidades según la clase a la que pertenecen o de la que son instancias.
* `iri`: filtra por el nombre o identificador de la entidad (admite comodines como *).
* `Propiedades directas` (`nombre_propiedad=valor`): filtra entidades que tengan un valor o relación específica asociada a una propiedad.

Ejemplos de uso:

```
# 1. Buscar todas las instancias que pertenecen a la clase 'VegetarianPizza'
pizzas_vegetarianas = onto.search(type=onto.VegetarianPizza)

# 2. Buscar todas las pizzas que contengan un ingrediente específico (GarlicTopping)
pizzas_con_ajo = onto.search(hasTopping=onto.GarlicTopping)

# 3. Buscar cualquier entidad cuyo nombre contenga la palabra "sauce"
salsas = onto.search(iri="*sauce*")
```

Una vez obtenida la lista de resultados de `search()`, se pueden recorrer sus elementos con un bucle e inspeccionar sus propiedades o clases con la sintaxis habitual de objetos en Python (por ejemplo, `entidad.name` o `entidad.is_a`).

**3. Control de inconsistencia lógica**
  * Inconsistencia por disyunción explícita de clases:
      *   Crear una nueva instancia `individuo_inconsistente`.
      * Asignarle simultáneamente dos clases declaradas como disjuntas en la TBox (por ejemplo, `Persona` y `Pizza`).

  * Inconsistencia por violación de reglas:
      * Crear una instancia declarada explícitamente como `VegetarianPizza`.
      * Añadirle un ingrediente perteneciente a la jerarquía de carnes `PepperoniTopping` o `MeatTopping`).

Ejecuta el razonador dentro de un bloque `try-except`. Capturar específicamente la excepción `OwlReadyInconsistentOntologyError` que lanza `Owlready2` cuando detecta un modelo incoherente, mostrando un mensaje explicativo por pantalla.

